# Colab Pro LLM Backend (Ollama + Pinggy.io)
Run this notebook on a Google Colab Pro instance with a hardware accelerator (T4, V100, or A100 GPU) enabled.
It sets up an OpenAI-compatible API endpoint exposing an Ollama model using Pinggy.io to bypass network firewalls.

In [1]:
# 1. Install require dependency for Ollama on new Ubuntu versions
!apt-get update -y && apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,385 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,787 kB] 
G

In [2]:
import os
import subprocess
import time

# 3. Tell Ollama to accept connections from anyone
os.environ["OLLAMA_HOST"] = "0.0.0.0"

# 3b. Compress the KV Cache to 8-bit to save massive amounts of VRAM.
# This is critical for achieving a 32,000 token window on an L4 GPU.
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"

# 4. Kill any stuck processes from previous runs
subprocess.run(["pkill", "-f", "ollama"])
time.sleep(2)

# 5. Start Ollama Server in the background
ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
print("✅ Ollama Server started in background.")

# 6. Give Ollama a few seconds to start up before pulling the model
time.sleep(3)

# 7. Pull the desired Model (We default to mistral, but can be llama3, etc.)
MODEL_NAME = "qwen2.5:32b" # YOU CAN CHANGE THIS
print(f"⏳ Pulling {MODEL_NAME}... this may take a minute or two depending on model size.")
subprocess.run(["/usr/local/bin/ollama", "pull", MODEL_NAME])
print(f"✅ Model {MODEL_NAME} pulled successfully.")

# 8. Start Pinggy Tunnel (Most reliable fallback)
print("\n" + "="*60)
print("⏳ Starting Pinggy Tunnel...")
pinggy_process = subprocess.Popen(
    ["ssh", "-p", "443", "-R0:localhost:11434", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "a.pinggy.io"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5) # Give it 5 seconds to establish the SSH handshake

url_found = False
for _ in range(20):
    line = pinggy_process.stdout.readline()
    if "http://" in line or "https://" in line:
        if "pinggy.link" in line:
            public_url = line.strip()
            if public_url.startswith("http://"):
                public_url = "https://" + public_url[7:] # Force HTTPS display
            print(f"🔥 PUBLIC TUNNEL URL: {public_url}")
            print("="*60 + "\n")
            print(f"👉 Copy the URL above and use `{public_url}/v1` as your base_url in your local IDE.")
            url_found = True
            break

if not url_found:
    print("❌ Could not extract Tunnel URL automatically. Check Colab logs for errors.")


✅ Ollama Server started in background.
⏳ Pulling qwen2.5:32b... this may take a minute or two depending on model size.
✅ Model qwen2.5:32b pulled successfully.

⏳ Starting Pinggy Tunnel...
🔥 PUBLIC TUNNEL URL: https://zolkx-34-16-178-80.a.free.pinggy.link

👉 Copy the URL above and use `https://zolkx-34-16-178-80.a.free.pinggy.link/v1` as your base_url in your local IDE.


## Keeping the Notebook Alive
Run this cell to prevent Colab from disconnecting while you use the API from your local IDE.

In [ ]:
import time
print("Entering infinite loop to keep instance alive...")
print("Press Stop to terminate.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped.")

Entering infinite loop to keep instance alive...
Press Stop to terminate.



Stopped.
